

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/opim-math/blob/main/OPIM5509/notebooks/BatchNorm_FromScratch.ipynb)

# BatchNorm From Scratch — What It Normalizes, and Where Its Parameters Are
**Dr. Dave Wanik** — Operations and Information Management — University of Connecticut

------------------------------------------


`model.summary()` shows a BatchNorm layer with **both** trainable *and* non-trainable parameters — the only layer that does. This notebook builds it in pure numpy so you can see exactly **what it computes, why it helps, and which numbers are learned by gradient descent vs. estimated from the data.**

## What it does
For each **feature** (column), across the current **batch**:
$$\mu=\text{mean},\quad \sigma^2=\text{var},\quad \hat x=\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}},\quad \text{out}=\gamma\,\hat x+\beta.$$
First it **standardizes** each feature to mean 0 / variance 1 (so no feature dominates and activations stay well-scaled), then it lets the network **undo or reshape** that with a learnable scale $\gamma$ and shift $\beta$.

In [ ]:
import numpy as np
x = np.array([1., 2., 3., 4.])          # one feature, a batch of 4
mu = x.mean(); var = x.var()
xhat = (x - mu) / np.sqrt(var + 1e-5)
print(f"mu = {mu}, var = {var}, sd = {np.sqrt(var):.3f}")
print(f"normalized x_hat = {np.round(xhat, 3)}   (mean 0, variance 1)")
gamma, beta = 1.0, 0.0                   # at init: identity
print(f"out = gamma*x_hat + beta = {np.round(gamma*xhat + beta, 3)}")

## Where the parameters live (the part students miss)
For a layer with $f$ features:

| parameter | count | how it's set | shows up as |
|---|---|---|---|
| $\gamma$ (scale), $\beta$ (shift) | $2f$ | **learned by gradient descent** | **trainable** |
| running mean, running variance | $2f$ | **exponential moving average of batch stats** | **non-trainable** |

So a `BatchNormalization()` over 8 features reports **16 trainable + 16 non-trainable** — and now you know why.

In [ ]:
f = 8
print(f"BatchNorm over {f} features:")
print(f"  trainable     = 2*f = {2*f}   (gamma, beta - learned)")
print(f"  non-trainable = 2*f = {2*f}   (running mean & var - tracked, not gradient-trained)")

## Train vs. inference — why there are *two* sets of statistics
- **During training** it normalizes using **this batch's** mean and variance, and quietly updates a *running* mean/variance ($\text{running}\leftarrow 0.99\cdot\text{running}+0.01\cdot\text{batch}$).
- **At inference** you often predict one sample at a time — a "batch" has no meaningful variance — so it uses the stored **running** statistics instead. That's why they must be saved, even though they aren't gradient-trained.

In [ ]:
# tiny training-loop sketch: watch the running stats track the data
run_mu, run_var, m = 0.0, 1.0, 0.99
for step, batch in enumerate([np.array([1.,2,3,4]), np.array([2.,3,4,5]), np.array([3.,4,5,6])]):
    run_mu  = m*run_mu  + (1-m)*batch.mean()
    run_var = m*run_var + (1-m)*batch.var()
    print(f"step {step}: batch mean={batch.mean():.2f}  ->  running mean now {run_mu:.4f}")
print("at inference we'd normalize with these running stats, not a single point's own mean.")

## Why it helps
- **Faster, more stable training:** keeping each layer's inputs well-scaled stops activations from drifting huge or tiny, so you can use higher learning rates.
- **Mild regularization:** each sample is normalized using *batch* statistics, injecting a little noise (a bit like dropout).
- **The catch:** behavior differs between train and inference — so always evaluate in inference mode (`model.eval()` / `training=False`).

```{admonition} ✋ Your turn
1. Compute the BatchNorm output of $x=[10, 20, 30]$ with $\gamma=2,\ \beta=1$ by hand, then check it here.
2. A network has `BatchNormalization()` over 64 features. How many trainable and non-trainable params? (Answer: 128 and 128.)
3. Why would BatchNorm with **batch size 1** be a bad idea during training?
```